# LC 746 — Min Cost Climbing Stairs
**Difficulty:** Easy | **Pattern:** 1D Dynamic Programming

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Each stair has a toll. You pay the
toll when you <em>stand</em> on that stair, then jump 1 or 2 steps.
The cheapest path to the top is built bottom-up: the minimum cost
to reach stair <code>i</code> equals its toll plus the cheaper of
the two stairs below it.
</div>

## Official Problem Statement

You are given an integer array `cost` where `cost[i]` is the cost
of the `i`-th step on a staircase. Once you pay the cost, you can
either climb one or two steps.

You can either start from the step with index `0`, or the step
with index `1`.

Return the **minimum cost** to reach the top of the floor.

**Constraints:**
- `2 <= cost.length <= 1000`
- `0 <= cost[i] <= 999`

## What This Is Actually Asking

Think of a staircase where every step has a coin toll. You pay
the toll, then hop 1 or 2 steps forward. "The top" is one step
beyond the last index — a virtual finish line with zero cost.
You want the cheapest sequence of hops that reaches that finish.
Because each decision only depends on the previous two stairs,
DP with two variables solves this in O(n) time and O(1) space.

## Walk Through an Example by Hand

`cost = [10, 15, 20]`

```
Step 0: toll=10
Step 1: toll=15
Step 2: toll=20
TOP  : (virtual, toll=0)
```

Build dp bottom-up:
```
dp[0] = 10          (start here, pay 10)
dp[1] = 15          (start here, pay 15)
dp[2] = 20 + min(dp[1], dp[0])
       = 20 + min(15, 10) = 30

Answer = min(dp[2], dp[1]) = min(30, 15) = 15
```

Path: start at step 1 (pay 15), jump 2 → top. Cost = **15**.

## The Picture

```
cost = [10, 15, 20]

Index:    0     1     2    TOP
Toll:    10    15    20     0
         |     |     |
dp:      10    15    30
                      \
                    min(dp[-1], dp[-2])
                    min(   30,     15) = 15

Recurrence:
  dp[0] = cost[0]
  dp[1] = cost[1]
  dp[i] = cost[i] + min(dp[i-1], dp[i-2])   for i >= 2
  answer = min(dp[-1], dp[-2])

Space-optimised (only two vars needed):
  prev2  prev1  cur
    10  →  15  →  30   ... slide window forward
```

## When To Use This Pattern

- When each position has a **local cost** and you choose 1 or 2
  steps forward, think **stair-step DP**.
- When the answer at index `i` depends only on `i-1` and `i-2`,
  think **two-variable rolling DP** (O(1) space).
- When the problem says "minimum/maximum cost to reach the end",
  think **bottom-up DP building toward the goal**.
- When the start point is flexible (index 0 or 1), think
  **initialise both base cases independently**.
- When constraints are small (n ≤ 1000), even O(n) extra space
  is fine, but rolling variables shows mastery.

## The Approach

Seed the DP with the first two tolls as base cases. Then sweep
forward: the cost to stand on stair `i` is `cost[i]` plus the
minimum cost to reach either of the two stairs below it. Track
only the last two values to keep space O(1). The answer is the
minimum of the last two DP values, because from either of those
stairs you can jump directly to the (virtual) top.

In [1]:
from typing import List

In [2]:
def test_harness(func):
    cases = [
        # (cost, expected)
        ([10, 15, 20],          15),
        ([1, 100, 1, 1, 1, 100, 1, 1, 100, 1], 6),
        ([0, 0],                 0),   # all zeros
        ([1, 2],                 1),   # two steps, pick cheaper
        ([5, 5],                 5),   # tie
        ([1, 1, 1, 1],           2),   # skip every other
    ]
    passed = 0
    for cost, expected in cases:
        result = func(cost)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            print(
                f"  {status}: cost={cost} "
                f"=> got {result}, want {expected}"
            )
    print(f"\nSummary: {passed}/{len(cases)} passed")

In [3]:
def min_cost_climbing_stairs(cost: List[int]) -> int:
    """
    Return the minimum cost to reach the top of the staircase.

    Strategy: 1D DP — roll two variables forward.
      dp[i] = cost[i] + min(dp[i-1], dp[i-2])
      answer = min(dp[-1], dp[-2])

    Args:
        cost: list of non-negative integers, len >= 2
    Returns:
        Minimum total toll paid on the way to the top.
    """
    dp = [0] * (len(cost)+1)
    for i in range(2, len(cost)+1):
        dp[i] = min(dp[i-2]+cost[i-2], dp[i-1] + cost[i-1])
    return dp[-1]
r'''
Summary: 6/6 passed
'''
test_harness(min_cost_climbing_stairs)



Summary: 6/6 passed


In [ ]:
# Uncomment and run when solution is ready
# test_harness(min_cost_climbing_stairs)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute-force recursion | O(2^n) | O(n) stack | Re-explores subproblems |
| Memoised recursion | O(n) | O(n) | Top-down DP |
| Bottom-up DP array | O(n) | O(n) | Simple to read |
| **Rolling two vars** | **O(n)** | **O(1)** | Optimal |

## Real World Connection

At **Citi**, fee-optimisation models choose between processing
paths where each node carries a transaction cost — the same
stair-step DP logic minimises total fees across a payment chain.
In **AWS Step Functions**, each Lambda invocation has a cost;
scheduling workflows to skip optional steps when they are pricey
mirrors choosing to jump two stairs. For a **data engineer**,
pipeline stages that can be skipped (e.g., an optional
deduplication step) present the same min-cost traversal problem.
This pattern appears whenever you have sequential decisions with
local costs and a fixed look-back of two positions.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra